# Notebook 3: Building machine learning models

In this part of the course, you will be introduced to molecular fingerprints and how these are used in similarity searches, clustering algorithms and machine learning. 

## Prepare yourself:

In order to make full use of this lessons, it is highly recommended to prepare yourselves by first having a look at the following movies:

- <a href="https://www.youtube.com/watch?v=xK9wHHQfGYA&list=LL&index=1" target="_blank">Machine learning and metrics</a> [YouTube - 24'48"]
- <a href="https://www.youtube.com/watch?v=TJveOYsK6MY" target="_blank">Supervised and unsupervised learning</a> [YouTube - 2'47"]
- <a href="https://www.youtube.com/watch?v=kBk8HbjWwCw" target="_blank">Molecular strings and fingerprints</a> [YouTube - 17'25"]

Then have a thorough read of these papers:

- <a href="assets/pdf/Molecular-fingerprint-similarity-search-in-virtual-screening.pdf" download>Molecular fingerprint similarity search in virtual screening</a> [pdf]
- <a href="assets/pdf/Applications-of-machine-learning-in-drug-discovery-and-development.pdf" download>Applications of machine learning in drug discovery and development</a> [pdf]


## Course manuals:

- <a href="assets/pdf/4-Fingerprints,_molecular_similarity_and_clustering.pdf" download>Chapter 4: Fingerprints, molecular similarity and clustering</a> [pdf]
- <a href="assets/pdf/5-QSAR_models.pdf" download>Chapter 5: QSAR models</a> [pdf]


## What you will learn

Notebooks 1 and 2 gave you a target and a way to write molecules down. Neither of those is a
number, and a computer cannot compare, cluster or learn from anything else. This notebook closes
that gap. A **fingerprint** turns a molecular graph into a vector of bits; once molecules are
vectors, similarity becomes arithmetic, clustering becomes geometry, and a model can be fitted that
predicts activity from structure.

Everything here is **ligand-based**: the protein never appears. The models learn from compounds
that are known to work, which is both the strength of the approach (you need no structure) and its
weakness (you can only find what resembles what you already know). Keep that trade-off in mind - in
notebook 5 you will attack the same problem from the structure side and see the difference.

At the end of this notebook you should be able to:

1. explain how a path-based, a circular (ECFP) and a key-based (MACCS) fingerprint are constructed,
   and what a bit collision is;
2. compute the Tanimoto and Tversky similarity between two molecules and interpret the number;
3. run a similarity search over a file of 10,000 compounds;
4. find the maximum common substructure of a set of molecules, and cluster molecules with both a
   hierarchical and a non-hierarchical algorithm;
5. train and validate a regression model that predicts pIC50 from a fingerprint, and explain why
   the result changes every time you re-split the data;
6. explain the difference between precision and recall, and decide which one matters for *your*
   question;
7. apply a trained classifier to a library of 100,000 compounds and judge what the hit list is
   worth.

## Planning (approximately 2 hours)

| Section | Content | Time |
|---|---|---|
| 1 | Fingerprints: path-based, circular and MACCS keys | 20 min |
| 2 | Measuring similarity, and a search through 10,000 compounds | 20 min |
| 3-4 | Maximum common substructure and clustering | 15 min |
| 5 | A first QSAR model: linear regression | 20 min |
| 6 | The same problem with a neural network | 10 min |
| 7 | Classification: precision versus recall | 25 min |
| 8 | Applying the model to 100,000 compounds | 10 min |

Two cells in section 7 search for optimal random-forest settings by brute force and take about a
minute each; the fingerprinting of the 100,000-compound library in section 8 takes about the same.
Everything else runs in seconds, so the two hours are meant to be spent thinking rather than
waiting.

## Install RDKit

In [ ]:
!pip install rdkit mols2grid requests

In [ ]:
# RDKit chemistry
from rdkit import Chem

# RDKit drawing
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import rdDepictor
IPythonConsole.ipython_useSVG = True
rdDepictor.SetPreferCoordGen(True)

# Library to display molecules in a grid
import mols2grid

# Library to download files
import requests

## 1. Molecular fingerprints

A **fingerprint** is a vector of bits that stands for a molecule. The recipe is always the same:
enumerate some kind of substructure, hash each one to a position in a bit vector, and set that bit.
What differs between the fingerprint families is *which* substructures are enumerated.

Because the number of possible substructures is far larger than the number of bits, two different
fragments can end up setting the same bit. That is a **bit collision**, and it is the price paid for
a fixed-length vector: a longer fingerprint collides less but costs more memory.

### 1.1 Linear path-based fingerprints

The RDKit fingerprint enumerates all linear **paths** through the molecular graph up to a maximum
length and hashes each of them.

**Question 1.** The first cell computes the same fingerprint at three different lengths, the second
one at four different maximum path lengths. Before running them: should the number of bits that are
ON go up or down as the fingerprint gets shorter, and why? Run the cells and explain what you see in
terms of bit collisions.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

mol = Chem.MolFromSmiles("Oc1ccc(CC(N)C(O)=O)cc1")

for fp_size in (10, 100, 1024):
  rdkgen = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=fp_size)
  fp = rdkgen.GetFingerprint(mol)
  print(len(list(fp.GetOnBits())), "bits ON out of the", len(fp), "bits in total")

mol

In [ ]:
mol = Chem.MolFromSmiles("Oc1ccc(CC(N)C(O)=O)cc1")

for max_path_length in (1,3,5,7):
  rdkgen = rdFingerprintGenerator.GetRDKitFPGenerator(maxPath=max_path_length)
  fp = rdkgen.GetFingerprint(mol)
  print(len(list(fp.GetOnBits())), "bits ON out of the", len(fp), "bits in total")

mol

### 1.2 Circular fingerprints (ECFP)

A circular or **Morgan** fingerprint does not follow paths but describes the **environment** of each
atom out to a given radius: radius 1 means the atom plus its direct neighbours, radius 2 adds the
next shell, and so on. ECFP4 - radius 2 - is the de-facto standard for similarity searching, and it
is the fingerprint most often used in published virtual screens.

**Question 2.** The cell loops over radii 1 to 7 for a small five-membered ring. The number of ON
bits rises and then stops rising. Why does it stop? (Think about how large the molecule is compared
with the radius.)

In [ ]:
mol = Chem.MolFromSmiles("O1CC(=O)NC1")

for radius in range(1,8):
  mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=1024)
  fp = mfpgen.GetFingerprint(mol)
  print("Radius", radius, ":", len(list(fp.GetOnBits())), "bits ON out of the", len(fp), "bits in total")

mol

### 1.3 MACCS keys

The MACCS set works the other way round. Rather than hashing whatever it finds, it asks 166 fixed,
hand-written questions: "is there a carboxylic acid?", "is there a halogen on an aromatic ring?".
Each bit therefore has a known chemical meaning, which makes a MACCS fingerprint interpretable -
but far less discriminating than a hashed fingerprint.

**Question 3.** You want to (a) find compounds similar to a known active in a million-compound
catalogue, and (b) explain to a medicinal chemist which structural features your model considers
important. Which fingerprint would you choose for each task, and why?

In [ ]:
from rdkit.Chem import MACCSkeys

mol = Chem.MolFromSmiles("Oc1ccc(CC(N)C(O)=O)cc1")
fp = MACCSkeys.GenMACCSKeys(mol)
print(len(list(fp.GetOnBits())), "bits ON out of the", len(fp), "bits in total")
print(list(fp.GetOnBits()))
mol

## 2. Calculating similarity

Once molecules are bit vectors, similarity is counting. The **Tanimoto** coefficient is the number
of bits that are on in both molecules divided by the number that are on in at least one:

$$T(A,B) = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

It runs from 0 to 1 and treats the two molecules symmetrically. The **Tversky** coefficient breaks
that symmetry on purpose: by weighting the two molecules differently you can ask "is A contained in
B?" rather than "are A and B alike?", which is what you want when the query is a small fragment and
the database contains large molecules.

As a rule of thumb, a Tanimoto above roughly 0.8 on a path-based fingerprint means the two compounds
are recognisably the same series - but that threshold depends entirely on the fingerprint, so treat
every such number with suspicion until you have calibrated it yourself.

In [ ]:
from rdkit import DataStructs

rdkgen = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=50)

mol1 = Chem.MolFromSmiles("CCOC")
fp1 = rdkgen.GetFingerprint(mol1)
print(fp1.ToBitString())

mol2 = Chem.MolFromSmiles("CCO")
fp2 = rdkgen.GetFingerprint(mol2)
print(fp2.ToBitString())

tanimoto = DataStructs.FingerprintSimilarity(fp1, fp2)
print(tanimoto)

In [ ]:
smiles = ["CO", "CCCO", "CCCOCCC"]
mols = []
for s in smiles: mols.append(Chem.MolFromSmiles(s))

rdkgen = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=50)
fps = []
for mol in mols: fps.append(rdkgen.GetFingerprint(mol))
ref = rdkgen.GetFingerprint(Chem.MolFromSmiles("CCCO"))

for fp in fps:
  tversky = DataStructs.TverskySimilarity(ref, fp, 0.1, 0.9)
  print("%.2f" % tversky)

print()
for fp in fps:
  tversky = DataStructs.TverskySimilarity(ref, fp, 0.9, 0.1)
  print("%.2f" % tversky)


**Question 4.** The cell above compares the same three molecules with the query `CCCO`, first with
Tversky weights (0.1, 0.9) and then with (0.9, 0.1). One of the two orderings says "the query is
contained in this molecule" and the other "this molecule is contained in the query". Which is
which? Why does `CCCOCCC` score so differently in the two runs, while `CCCO` itself does not?

### 2.1 Similarity in practice: searching 10,000 compounds

This is virtual screening in its simplest form: one query, one database, one number per compound,
and the highest number wins.

In this exercise, a file with 10,000 compounds (SMILES format) is downloaded from the UAMC GitHub repo, and then a similarity search is performed to identify the compound that is most similar to aspirin ("CC(=O)OC1=CC=CC=C1C(=O)O")

In [ ]:
url = "https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/master/assets/databases/notebook-3/compounds_10k.smi"
smiles = requests.get(url).text.split("\n")
print(len(smiles))

Aspirin is the query molecule:

In [ ]:
rdkgen = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)
aspirin = Chem.MolFromSmiles("CC(=O)OC1=CC=CC=C1C(=O)O")
query = rdkgen.GetFingerprint(aspirin)
aspirin

Loop over all molecules and keep the one with the highest similarity to aspirin:

In [ ]:
max_sim = 0.0
best_hit = ""
for s in smiles:
  mol = Chem.MolFromSmiles(s)
  fp = rdkgen.GetFingerprint(mol)
  tanimoto = DataStructs.FingerprintSimilarity(fp, query)
  #print(tanimoto)    # uncomment to watch all 10,000 values go by (slow)
  if tanimoto >= max_sim:
    max_sim = tanimoto
    best_hit = s

Show the best molecule:

In [ ]:
mol = Chem.MolFromSmiles(best_hit)
print(max_sim)
mol

Note: you can also use other similarity metrics such as the Tversky coefficient. Try this out to see how the results will alter.

**Question 5.** Look at the best hit. Is it chemically similar to aspirin in the way *you* would
judge similarity, or only in the way the fingerprint does? Now the important question for the rest
of the course: a fingerprint search compares substructures the query already has. Could this search
ever return a compound that binds the same target through a completely different scaffold? Keep
your answer; notebook 5 attacks precisely this limitation.

## 3. Maximum common substructure

Similarity gives you a number; the **maximum common substructure** gives you a picture. It is the
largest fragment that a set of molecules share, and it is the natural way to answer "what do these
actives have in common?". Below it is applied to three opiates, which should produce a recognisable
morphinan skeleton.

In [ ]:
from rdkit.Chem import rdFMCS

morphine = Chem.MolFromSmiles("CN1CC[C@]23C4=C5C=CC(O)=C4O[C@H]2[C@H](C=C[C@H]3[C@H]1C5)O")
codeine = Chem.MolFromSmiles("CN1CC[C@]23[C@@H]4[C@H]1CC5=C2C(O[C@H]3[C@@H](O)C=C4)=C(OC)C=C5")
heroine = Chem.MolFromSmiles("CN([C@H](CC(C=C1)=C23)[C@@H]4C=C[C@@H]5OC(C)=O)CC[C@]43[C@H]5OC2=C1OC(C)=O")

mols = [morphine, codeine, heroine]
mcss = rdFMCS.FindMCS(mols)
Chem.MolFromSmarts(mcss.smartsString)

## 4. Clustering

**Clustering** groups molecules by their similarity without being told what the groups are - it is
*unsupervised*. Two families of algorithm are shown here. **Hierarchical** clustering repeatedly
merges the two closest groups, so you choose how far down the tree to cut. **k-means** instead
places a chosen number of centres and assigns every molecule to the nearest one.

In practice clustering is how a hit list is turned into a purchase list: cluster the hits, buy one
or two compounds per cluster, and you cover the chemical space with the fewest compounds. You will
have exactly that problem at the end of notebook 4.

Read in six molecules:

In [ ]:
smiles = ["c1ccccc1", "c1cccnc1", "c1ncncc1", "C1CC1", "CC=O", "NCC"]
mols = [Chem.MolFromSmiles(x) for x in smiles]

mfpgen = rdFingerprintGenerator.GetMorganGenerator(fpSize=20)
fps = [mfpgen.GetFingerprint(mol) for mol in mols]
display(Draw.MolsToGridImage(mols, molsPerRow=3))

Show their fingerprints:

In [ ]:
import numpy as np
for i in range(len(fps)): print("%s %s" % (fps[i].ToBitString(), smiles[i]))

Convert the fingerprints to a format that is useable by a clustering algorithm:

In [ ]:
nps = [np.array(x) for x in fps]
X = np.array(nps)
print(X)

Do an hierarchical clustering:

In [ ]:
import sklearn
from sklearn.cluster import AgglomerativeClustering
clusterEngine = AgglomerativeClustering(n_clusters = 3)
clusterEngine.fit(X)

labels = [str(x) for x in clusterEngine.labels_]
display(Draw.MolsToGridImage(mols, molsPerRow=3, legends=labels))

And now non-hierarchical clustering (k-means):

**Question 6.** Six molecules, three clusters, two algorithms. Do the two methods agree? Look at the
molecules themselves: is the grouping chemically sensible, and which molecule is hardest to place?
What would change if you asked for two clusters instead of three?

In [ ]:
from sklearn.cluster import KMeans
clusterEngine = KMeans(n_clusters = 3)
clusterEngine.fit(X)

labels = [str(x) for x in clusterEngine.labels_]
display(Draw.MolsToGridImage(mols, molsPerRow=3, legends=labels))

In [ ]:
centers = clusterEngine.cluster_centers_
print(centers)

# Machine learning: QSAR models

## 5. A first model: linear regression

A **QSAR** model predicts a property from structure. The ingredients are a set of compounds with
measured activities, a fingerprint for each of them, and a function fitted to map the one onto the
other. The measured quantity here is pIC50, the negative logarithm of the IC50: a pIC50 of 9 is a
1 nM inhibitor, a pIC50 of 6 is a 1 uM inhibitor, and - because the scale is logarithmic - an error
of one unit in a prediction is a factor of ten in potency.

Read in a dataset of DPP4 inhibitors with corresponding pIC50 inhibition constants:

In [ ]:
url = "https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/master/assets/databases/notebook-3/dpp4.pIC50.txt"
data = requests.get(url).text.split("\n")
print(data[0])

Split into smiles, mols, fps and pIC50:

In [ ]:
rdkgen = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)

mols = []
smiles = []
fps = []
pic50 = []
for d in data:
  fields = d.split()
  if len(fields) < 1: continue
  smiles.append(fields[0])
  pic50.append(float(fields[1]))
  mol = Chem.MolFromSmiles(fields[0])
  mols.append(mol)
  fp = np.zeros((0,), dtype=np.int8)
  DataStructs.ConvertToNumpyArray(rdkgen.GetFingerprint(mol), fp)
  fps.append(fp)
print(smiles[0])
print(pic50[0])
print(fps[0])
print(max(pic50))
print(min(pic50))
print(len(smiles))

Create a training set (70%) and a test set (30%):

In [ ]:
from sklearn.model_selection import train_test_split

pic50_train, pic50_test, fps_train, fps_test = train_test_split(pic50, fps, test_size=0.3, random_state=42)
print(len(pic50_train), len(pic50_test))

Train a linear regression model:

In [ ]:
from sklearn import linear_model

model = linear_model.LinearRegression()
model.fit(fps_train, pic50_train)
print(model.coef_)

Apply the trained model on the test set and compare the predicted values with the experimental ones:

In [ ]:
pic50_pred = model.predict(fps_test)
print(pic50_pred)

Validate the model by calculating the MSE of the predictions when compared to the true values:

In [ ]:
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

print("MSE = ", mean_squared_error(pic50_test, pic50_pred))
plt.plot(pic50_test, pic50_pred, '.')
plt.xlabel("True values")
plt.ylabel("Predicted values")

Repeat the test/train splitting a number of times in order to get statistics:

**Question 7.** Each repetition gives a different MSE, sometimes very different. Where does that
variation come from? Note the shape of the data: roughly 2,700 training molecules described by 2,048
features. What is the name for the problem that this ratio invites, and which of the two models you
have now seen - linear regression or the neural network - is more exposed to it?

In [ ]:
for i in range(5):
  pic50_train, pic50_test, fps_train, fps_test = train_test_split(pic50, fps, test_size=0.3)
  model.fit(fps_train, pic50_train)
  pic50_pred = model.predict(fps_test)
  print("MSE = ", mean_squared_error(pic50_test, pic50_pred))

## 6. A more complicated model: a neural network

A linear model can only add up weighted contributions of individual bits. A neural network puts one
or more hidden layers in between, which lets it represent combinations of features - "this fragment
*together with* that one". It has more parameters than the linear model and therefore more
opportunity to memorise the training set, so honest validation matters even more.

In [ ]:
from sklearn.neural_network import MLPRegressor
model = MLPRegressor(random_state=1, max_iter=500)
model.fit(fps_train, pic50_train)
pic50_pred = model.predict(fps_test)
print("MSE =", mean_squared_error(pic50_test, pic50_pred))
plt.plot(pic50_test, pic50_pred, '.')
plt.xlabel("True values")
plt.ylabel("Predicted values")

Repeat the test/train splitting a number of times in order to get statistics:

In [ ]:
for i in range(3):
  pic50_train, pic50_test, fps_train, fps_test = train_test_split(pic50, fps, test_size=0.3)
  model.fit(fps_train, pic50_train)
  pic50_pred = model.predict(fps_test)
  print("MSE =", mean_squared_error(pic50_test, pic50_pred))

Now built a model using the entire dataset and save for later on:

In [ ]:
pic50_predictor = MLPRegressor(max_iter=500)
pic50_predictor.fit(fps, pic50)

## 7. Classification: random forests, precision and recall

So far we predicted a number. Often the real question is a decision: is this compound worth testing,
yes or no? That is **classification**, and the quality of a classifier cannot be captured by a
single number - which is what the rest of this section is about.

A **random forest** fits many decision trees, each on a random part of the data and of the features,
and lets them vote. That makes it robust, fast and a sensible default for fingerprint data.

Load a DPP4 dataset with actives and non-actives (classification model):

In [ ]:
url = "https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/master/assets/databases/notebook-3/dpp4.classified.txt"
data = requests.get(url).text.split("\n")
print(data[0])

Generate fingerprints and make a list of all the activities:

In [ ]:
rdkgen = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)

activities = []
fps = []
for d in data:
  if d is None or d == "": continue
  fields = d.split()
  if fields[1] == "ACTIVE": activities.append(1)
  if fields[1] == "INACTIVE": activities.append(0)
  mol = Chem.MolFromSmiles(fields[0])
  fp = np.zeros((0,), dtype=np.int8)
  DataStructs.ConvertToNumpyArray(rdkgen.GetFingerprint(mol), fp)
  fps.append(fp)

print(len(activities), len(fps))

Let's count how many actives and inactives:

In [ ]:
n_actives = 0
n_inactives = 0
for a in activities:
  if a == 1: n_actives += 1
  if a == 0: n_inactives += 1
print(f'Actives: {n_actives}')
print(f'Inactives: {n_inactives}')

Train a random forest model to predict activity:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

act_train, act_test, fps_train, fps_test = train_test_split(activities, fps, test_size=0.3)
model = RandomForestClassifier(max_depth=2)
model.fit(fps_train, act_train)

Calculate the accuracy of the generated model:

In [ ]:
from sklearn.metrics import accuracy_score

prediction = model.predict(fps_test)
print(accuracy_score(act_test, prediction))

**Precision versus recall**

The **precision** metric is defined by TP / (TP + FP). A high precision is therefore obtained when the model **reduces the number of false positives**. A model with a high precision is useful in a situation where one wants to be sure that, if the model predicts a compound to be active, the compound is really active, however at the expense of missing some other actives. This might be useful when one wants to purchase compounds with a limited budget.

The **recall** metric is defined by TP / (TP + FN). A high recall is therefore obtained when the model **reduces the number of false negatives**. A model with a high recall is therefore useful in a situation where one wants to be sure that **all** active compounds in a database will be purchased, however at the expense of buying also compounds which are not active. This might be useful when one can purchase compounds with an unlimited budget.

Let us make a random forest model with a **high precision**. We can do this by tweaking some of the parameters in the random forest classifier. The main parameters to adjust when using these methods is *n_estimators* and *max_features*. The former is the number of trees in the forest. The larger the better, but also the longer it will take to compute. The latter is the size of the random subsets of features to consider when splitting a node. The lower the greater the reduction of variance, but also the greater the increase in bias.

In [ ]:
from sklearn.metrics import precision_score

optimal_max_depth = 0
optimal_n_estimators = 0
optimal_max_features = 0
optimal_precision = 0
for max_depth in range(4,10):
  print(f'Max_depth: {max_depth}')
  for n_estimators in range(4,10):
    for max_features in range(4,10):
      precision = []
      for i in range(5): # Repeat 5 times to get average
        act_train, act_test, fps_train, fps_test = train_test_split(activities, fps, test_size=0.3)
        model = RandomForestClassifier(max_depth=max_depth, n_estimators=n_estimators, max_features=max_features)
        model.fit(fps_train, act_train)
        prediction = model.predict(fps_test)
        precision.append(precision_score(act_test, prediction))
      mean_precision = np.mean(precision)
      if mean_precision > optimal_precision:
        optimal_max_depth = max_depth
        optimal_n_estimators = n_estimators
        optimal_max_features = max_features
        optimal_precision = mean_precision
print(f'max_depth: {optimal_max_depth}, n_estimators: {optimal_n_estimators}, max_features: {optimal_max_features} -> precision = {optimal_precision}')

Now that we have the optimal parameters for a **high precision** model, let's train this model and count the number of TP's and FP's in the test set:

This brute-force search over `max_depth`, `n_estimators` and `max_features` fits roughly a thousand
forests and takes about a minute.

In [ ]:
from sklearn.metrics import confusion_matrix

TP = []
FP = []
FN = []
for i in range(10): # Repeat 10 times to get average
  act_train, act_test, fps_train, fps_test = train_test_split(activities, fps, test_size=0.3)
  precision_model = RandomForestClassifier(max_depth=optimal_max_depth, n_estimators=optimal_n_estimators, max_features=optimal_max_features)

  precision_model.fit(fps_train, act_train)
  prediction = precision_model.predict(fps_test)

  tn, fp, fn, tp = confusion_matrix(act_test, prediction).ravel().tolist()

  TP.append(tp)
  FP.append(fp)
  FN.append(fn)

print(f'TP: {np.mean(TP)}')
print(f'FP: {np.mean(FP)}')
print(f'FN: {np.mean(FN)}')


Let us make a random forest model with a **high recall**:

In [ ]:
from sklearn.metrics import recall_score

optimal_max_depth = 0
optimal_n_estimators = 0
optimal_max_features = 0
optimal_recall = 0
for max_depth in range(4,10):
  print(f'Max_depth: {max_depth}')
  for n_estimators in range(4,10):
    for max_features in range(4,10):
      recall = []
      for i in range(5): # Repeat 5 times to get average
        act_train, act_test, fps_train, fps_test = train_test_split(activities, fps, test_size=0.3)
        model = RandomForestClassifier(max_depth=max_depth, n_estimators=n_estimators, max_features=max_features)
        model.fit(fps_train, act_train)
        prediction = model.predict(fps_test)
        recall.append(recall_score(act_test, prediction))
      mean_recall = np.mean(recall)
      if mean_recall > optimal_recall:
        optimal_max_depth = max_depth
        optimal_n_estimators = n_estimators
        optimal_max_features = max_features
        optimal_recall = mean_recall
print(f'max_depth: {optimal_max_depth}, n_estimators: {optimal_n_estimators}, max_features: {optimal_max_features} -> recall = {optimal_recall}')

Now that we have the optimal parameters for a high recall model, let's train this model and count the number of TP's and FP's in the test set:

**Question 8.** Compare the TP, FP and FN counts of the high-precision model with those of the
high-recall model. You have a budget for 50 compounds and a single shot at the assay: which model do
you order from? Your colleague can test 5,000 compounds cheaply and wants to miss nothing: which
model do they want? Is there a setting that is best for both of you?

In [ ]:
TP = []
FP = []
FN = []
for i in range(10): # Repeat 10 times to get average
  act_train, act_test, fps_train, fps_test = train_test_split(activities, fps, test_size=0.3)
  recall_model = RandomForestClassifier(max_depth=optimal_max_depth, n_estimators=optimal_n_estimators, max_features=optimal_max_features)

  recall_model.fit(fps_train, act_train)
  prediction = recall_model.predict(fps_test)

  tn, fp, fn, tp = confusion_matrix(act_test, prediction).ravel().tolist()

  TP.append(tp)
  FP.append(fp)
  FN.append(fn)

print(f'TP: {np.mean(TP)}')
print(f'FP: {np.mean(FP)}')
print(f'FN: {np.mean(FN)}')

## 8. Applying the model to a library of 100,000 compounds

This is the pay-off: a model trained on a few thousand measured compounds is now asked about a
hundred thousand it has never seen. Fingerprinting the library takes about a minute.

In [ ]:
url = "https://raw.githubusercontent.com/UAMCAntwerpen/2040FBDBIC/master/assets/databases/notebook-3/compounds_100k.smi"
data = requests.get(url).text.split("\n")
print(data[0])

Calculate fingerprints:

In [153]:
rdkgen = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)

db_fps = []
for d in data:
  if d == "" or d is None: continue
  mol = Chem.MolFromSmiles(d)
  fp = np.zeros((0,), dtype=np.int8)
  DataStructs.ConvertToNumpyArray(Chem.RDKFingerprint(mol), fp)
  db_fps.append(fp)

Apply the RF on the database and extract those that are predicted to be active:

In [ ]:
prediction = recall_model.predict(db_fps)
hits_smiles = []
hits_fps = []
for i in range(len(prediction)):
  if prediction[i] == 1:
    hits_smiles.append(data[i])
    hits_fps.append(db_fps[i])
print(len(hits_smiles), len(hits_fps))

Score the identified hits with the pIC50 neural network model:

In [ ]:
hits_pic50 = pic50_predictor.predict(hits_fps)

Identify the compound with the highest pIC50 and show structure:

**Question 9.** The classifier selected a few thousand compounds out of 100,000, and the regression
model then picked the most potent one. Two models, two predictions, one molecule. What evidence do
you actually have that this compound inhibits DPP4? What would you want to do before spending money
on it?

In [ ]:
best_pic50 = 0
best_smiles = ""
for i in range(len(hits_pic50)):
  if hits_pic50[i] > best_pic50:
    best_pic50 = hits_pic50[i]
    best_smiles = hits_smiles[i]
print(best_pic50)
print(best_smiles)
mol = Chem.MolFromSmiles(best_smiles)
mol

## Finished?

You can now turn a molecule into a vector, compare vectors, group them, and fit a model that maps
them onto an activity. You have also met the two honest limitations of doing so: a model is only as
good as the compounds it learned from, and a fingerprint can only recognise what it has seen before.

For those that are interested in machine learning for drug discovery, this paper provides an
excellent review of the current state-of-the-art on this topic:

- <a href="assets/pdf/Machine-learning-in-drug-discovery-a-review.pdf" download>Machine learning in drug discovery: a review</a>

When finished, it is time to move to the fourth notebook of this course. There you will apply
everything above to the MDM2-p53 system and build a focussed database of candidate binders:

- [Notebook 4: Building a focussed database of putative MDM2 binders](Notebook-4-Building-database-of-MDM2-binders.ipynb)